In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import logging
import joblib

# Sklearn / Imblearn
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)
from imblearn.combine import SMOTETomek

# Feature Selection & Models
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def train_with_rfecv(csv_path="compiled_features2.csv",
                     save_model_path="best_model_rfecv.pkl"):
    """
    1. Loads feature DataFrame from a CSV file.
    2. Applies train-test split and SMOTETomek to handle imbalance.
    3. Uses a Pipeline with StandardScaler -> RFECV(LogisticRegression).
    4. Tunes hyperparameters using RandomizedSearchCV.
    5. Evaluates best model and saves it to a pickle file.
    6. Prints selected features from RFECV.
    """

    ############################################################################
    # 1) Load Data
    ############################################################################
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"[ERROR] CSV file not found: {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"[INFO] Loaded data shape: {df.shape}")

    # Basic check for the label column
    if "Label" not in df.columns:
        raise ValueError("[ERROR] 'Label' column not found in the CSV.")

    # Drop non-feature columns if present
    # e.g. 'Label', 'Datetime', 'Size', 'No'
    drop_cols = ['Label', 'Datetime', 'Size', 'No']
    feature_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feature_cols]
    y = df['Label']

    # Clean infinite / NaN
    X = X.replace([np.inf, -np.inf], np.nan)
    if X.isnull().any().any():
        # Drop rows that have NaNs
        null_rows = X.isnull().any(axis=1)
        X = X[~null_rows]
        y = y[~null_rows]

    ############################################################################
    # 2) Train-Test Split
    ############################################################################
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        stratify=y,
        random_state=42
    )
    print("[INFO] Train shape:", X_train.shape, "Test shape:", X_test.shape)

    ############################################################################
    # 3) Handle Class Imbalance
    ############################################################################
    smt = SMOTETomek(random_state=42)
    X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
    print("[INFO] After SMOTETomek, X_train_res shape:", X_train_res.shape)

    ############################################################################
    # 4) Define Pipeline with RFECV
    ############################################################################
    # RFECV itself contains an "estimator" (LogisticRegression) and does
    # repeated fitting to determine the optimal subset of features.
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('rfecv', RFECV(
            estimator=LogisticRegression(random_state=42, max_iter=2000),
            step=1,                # how many features to remove at each step
            cv=5,                  # internal cross-validation
            scoring='accuracy',    # metric used to evaluate feature subsets
            min_features_to_select=1  # minimum number of features to keep
        ))
    ])

    ############################################################################
    # 5) Hyperparameter Tuning with RandomizedSearchCV
    ############################################################################
    param_dist = {
        # Tune the logistic regression inside RFECV
        'rfecv__estimator__C': [0.001, 0.01, 0.1, 1, 10],
        # Possibly tune the step size in RFECV
        'rfecv__step': [1, 2],
        # Optionally tune min_features_to_select
        'rfecv__min_features_to_select': [1, 5, 10]
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    rand_search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        cv=cv,
        scoring='accuracy',
        n_iter=10,       # Increase for a more thorough search
        n_jobs=-1,
        verbose=1,
        random_state=42,
        refit=True
    )

    print("[INFO] Starting RandomizedSearchCV...")
    rand_search.fit(X_train_res, y_train_res)

    best_model = rand_search.best_estimator_
    print("[INFO] Best Parameters:", rand_search.best_params_)
    print("[INFO] Best CV Score (Accuracy):", f"{rand_search.best_score_:.4f}")

    ############################################################################
    # 6) Evaluate on Test Set
    ############################################################################
    y_pred = best_model.predict(X_test)
    # If logistic regression, we have predict_proba:
    if hasattr(best_model.named_steps['rfecv'].estimator, "predict_proba"):
        y_proba = best_model.named_steps['rfecv'].estimator.predict_proba(
            best_model.named_steps['scaler'].transform(X_test)
        )[:, 1]
    else:
        # fallback if no predict_proba
        y_proba = y_pred.astype(float)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1)
    rec = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    roc_auc = roc_auc_score(y_test, y_proba)

    print("\n[RESULTS] Test Set Performance:")
    print(f"  Accuracy:   {acc:.2f}")
    print(f"  Precision:  {prec:.2f}")
    print(f"  Recall:     {rec:.2f}")
    print(f"  F1 Score:   {f1:.2f}")
    print(f"  ROC AUC:    {roc_auc:.2f}")

    # Classification Report
    report_dict = classification_report(y_test, y_pred, target_names=['Failed','Passed'], output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    print("\n[Classification Report]")
    print(report_df[['precision', 'recall', 'f1-score', 'support']].round(2))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Failed','Passed'],
                yticklabels=['Failed','Passed'])
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc(fpr, tpr):.2f})', color='darkorange')
    plt.plot([0,1],[0,1], linestyle='--', color='navy')
    plt.xlim([0,1])
    plt.ylim([0,1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

    ############################################################################
    # 7) Inspect Which Features Were Selected
    ############################################################################
    # RFECV is the second step in the pipeline
    rfecv_step = best_model.named_steps['rfecv']
    # 'support_' is a boolean mask of selected features
    feature_mask = rfecv_step.support_
    selected_features = X_train.columns[feature_mask]
    print("\n[INFO] Number of features selected:", sum(feature_mask))
    print("[INFO] Selected features:")
    for feat in selected_features:
        print("   ", feat)

    ############################################################################
    # 8) Save the Best Model
    ############################################################################
    joblib.dump(best_model, save_model_path)
    print(f"\n[INFO] Best model saved to '{save_model_path}'")

    return best_model


if __name__ == "__main__":
    # Example usage
    best_model = train_with_rfecv(
        csv_path=r"C:\Users\chimpaleenantaphon\Documents\datafortraining - Copy\compiled_features2.csv",
        save_model_path="best_model_rfecv.pkl"
    )
